# 🧠 BrainAgenticthon-2026 — Dataset Access & Validation

This notebook runs **inside Google Colab** to acquire and validate 3D brain
MRI datasets for the BrainAgenticthon challenge.

**Prerequisites:**
- A [Synapse](https://www.synapse.org/) account with approved access to the BraTS dataset (or another authorised source)
- Your `.synapseConfig` file uploaded to Google Drive (private, never shared)

**What this notebook does:**
1. Mounts Google Drive for persistent storage
2. Installs dependencies and clones the repo
3. Configures dataset sources via the pluggable registry
4. Supports dry-run inspection before any download
5. Fetches data to Colab's ephemeral `/content` workspace
6. Validates NIfTI integrity, modalities, case structure, and label maps
7. Saves non-sensitive outputs to Google Drive

> ⚠️ **Security:** Never print credentials, tokens, or patient identifiers in cell outputs.

---
## 1 — Environment Setup

### 1.1 Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# Verify mount
import os
assert os.path.isdir("/content/drive/MyDrive"), "Google Drive not mounted!"
print("✅ Google Drive mounted successfully.")

### 1.2 Create Persistent Project Directory on Drive

In [ ]:
DRIVE_PROJECT_ROOT = "/content/drive/MyDrive/BrainAgenticthon-2026"

for subdir in ["checkpoints", "metrics", "outputs", "logs", "configs_snapshots"]:
    os.makedirs(os.path.join(DRIVE_PROJECT_ROOT, subdir), exist_ok=True)

print(f"✅ Persistent storage ready at: {DRIVE_PROJECT_ROOT}")
!ls -la {DRIVE_PROJECT_ROOT}

### 1.3 Install Dependencies & Clone Repository

In [ ]:
# Install required packages (nibabel for NIfTI, synapseclient for BraTS)
!pip install -q nibabel synapseclient pyyaml

print("✅ Dependencies installed.")

In [ ]:
REPO_URL = "https://github.com/abdulrahmantaani24-dotcom/BrainAgenticthon-2026.git"
COLAB_WORKSPACE = "/content/brainagenticthon"

if not os.path.isdir(COLAB_WORKSPACE):
    !git clone {REPO_URL} {COLAB_WORKSPACE}
else:
    !cd {COLAB_WORKSPACE} && git pull --ff-only

# Install the package in editable mode
!pip install -q -e {COLAB_WORKSPACE}

print(f"✅ Repository ready at: {COLAB_WORKSPACE}")

### 1.4 Verify Package Import

In [ ]:
import sys
sys.path.insert(0, os.path.join(COLAB_WORKSPACE, "src"))

import brainagenticthon
print(f"✅ brainagenticthon v{brainagenticthon.__version__} imported successfully.")

from brainagenticthon.preprocessing.dataset_registry import (
    DatasetRegistry,
    StorageConfig,
    is_colab,
)
from brainagenticthon.preprocessing.data_validator import (
    NIfTIValidator,
    CaseValidator,
    DatasetValidator,
)

print(f"Running in Colab: {is_colab()}")
print("✅ All modules imported.")

---
## 2 — Configure Storage

In [ ]:
# Load storage configuration
# Option A: from the example config in the repo
config_path = os.path.join(COLAB_WORKSPACE, "configs", "storage.example.yaml")

# Option B: from your private config on Google Drive (recommended)
# config_path = os.path.join(DRIVE_PROJECT_ROOT, "configs_snapshots", "storage.yaml")

cfg = StorageConfig.from_yaml(config_path)
print(f"Workspace:    {cfg.workspace}")
print(f"Checkpoints:  {cfg.resolve_path('checkpoints')}")
print(f"Metrics:      {cfg.resolve_path('metrics')}")
print(f"Outputs:      {cfg.resolve_path('outputs')}")
print("✅ Storage configured.")

---
## 3 — Dataset Registry & Source Selection

In [ ]:
# List all registered dataset sources
registry = DatasetRegistry()

print("Available dataset sources:")
print("-" * 50)
for source_name in registry.list_sources():
    source = registry.get_source(source_name)
    print(f"  • {source_name}: {source.description}")

### 3.1 Configure BraTS Source (via Synapse)

**Before running this cell**, ensure you have:
1. A Synapse account with BraTS data access approved
2. Your `.synapseConfig` uploaded to Google Drive (private folder)

If you don't have access yet, skip to Section 3.2 (Local Directory Source).

In [ ]:
from brainagenticthon.preprocessing.dataset_registry import BraTSSource

# ===========================================================================
# ⚠️  EDIT THESE VALUES — replace with your actual Synapse project ID
#    and the path to your .synapseConfig on Google Drive.
# ===========================================================================
SYNAPSE_PROJECT_ID = "syn00000000"   # <-- Replace with your actual project ID
SYNAPSE_CONFIG     = "/content/drive/MyDrive/.private/.synapseConfig"

brats_source = BraTSSource(
    synapse_project_id=SYNAPSE_PROJECT_ID,
    credentials_path=SYNAPSE_CONFIG,
)

# Show source info (does NOT print credentials)
print(f"Source:      {brats_source.name}")
print(f"Description: {brats_source.description}")
print(f"Modalities:  {brats_source.required_modalities}")

### 3.2 Alternative: Local Directory Source

Use this if you have already downloaded or transferred the dataset manually.

In [ ]:
from brainagenticthon.preprocessing.dataset_registry import LocalDirectorySource

# Point to a directory already containing NIfTI files
# local_source = LocalDirectorySource(
#     root_dir="/content/drive/MyDrive/datasets/my_brain_mri",
#     modalities=["t1", "t1ce", "t2", "flair", "seg"],
# )
# print(f"Source: {local_source.name}")

print("(Uncomment the above lines and set your local dataset path.)")

---
## 4 — Dry-Run Inspection

**Always run a dry-run first** to see what would be downloaded, without
actually fetching any data.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(name)s | %(message)s")

OUTPUT_DIR = os.path.join(COLAB_WORKSPACE, "data", "raw", "brats")

# ---- DRY RUN ----
print("=" * 60)
print("DRY RUN — nothing will be downloaded")
print("=" * 60)

brats_source.fetch(output_dir=OUTPUT_DIR, dry_run=True)

print("\n✅ Dry run complete. Review the above before proceeding.")

---
## 5 — Fetch Data (Actual Download)

> ⚠️ **Only run this after reviewing the dry-run output above.**
>
> This cell downloads data from Synapse to the Colab ephemeral workspace.
> It does **not** upload anything to GitHub.

In [ ]:
# ⚠️  Uncomment the lines below ONLY after reviewing the dry-run output.
#     This will download data to the Colab workspace.

# print("Fetching dataset — this may take a while...")
# brats_source.fetch(output_dir=OUTPUT_DIR, dry_run=False)
# print("✅ Download complete.")

print("(Uncomment the above lines to start the actual download.)")

---
## 6 — Validate Downloaded Data

### 6.1 Validate a Single NIfTI File

In [ ]:
# Example: validate one NIfTI file
# Uncomment and set the path to an actual file after downloading

# validator = NIfTIValidator()
# report = validator.validate_all("/path/to/a/volume.nii.gz")
# print(report)

print("(Set the path to a real NIfTI file after downloading data.)")

### 6.2 Validate a Single Case

In [ ]:
# Example: validate one case directory

# case_validator = CaseValidator()
# report = case_validator.validate_case_directory(
#     case_dir="/content/brainagenticthon/data/raw/brats/BraTS-GLI-00000-000",
#     required_modalities=["t1", "t1ce", "t2", "flair", "seg"],
# )
# print(report)

print("(Set the case directory path after downloading data.)")

### 6.3 Validate Entire Dataset

In [ ]:
# Full dataset validation (runs all checks on every case)

# dataset_validator = DatasetValidator()
# report = dataset_validator.generate_report(
#     dataset_dir=OUTPUT_DIR,
#     required_modalities=["t1", "t1ce", "t2", "flair", "seg"],
#     expected_labels=[0, 1, 2, 3],
# )
# print(report)
#
# # Save report to Drive for later reference
# import json
# report_path = os.path.join(DRIVE_PROJECT_ROOT, "logs", "validation_report.json")
# with open(report_path, "w") as f:
#     json.dump(report.to_dict(), f, indent=2)
# print(f"Report saved to: {report_path}")

print("(Uncomment after downloading data to run full validation.)")

---
## 7 — Save Experiment Config Snapshot

Freeze the current configuration to Google Drive for reproducibility.

In [ ]:
import shutil
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
snapshot_dir = os.path.join(DRIVE_PROJECT_ROOT, "configs_snapshots")
os.makedirs(snapshot_dir, exist_ok=True)

# Copy the config used in this session
snapshot_path = os.path.join(snapshot_dir, f"storage_{timestamp}.yaml")
shutil.copy2(config_path, snapshot_path)

print(f"✅ Config snapshot saved: {snapshot_path}")

---
## 8 — Summary & Next Steps

| Item | Location | Persistent? |
|------|----------|-------------|
| Raw MRI data | `/content/brainagenticthon/data/raw/` | ❌ Ephemeral |
| Checkpoints | Google Drive `checkpoints/` | ✅ Yes |
| Metrics | Google Drive `metrics/` | ✅ Yes |
| Config snapshots | Google Drive `configs_snapshots/` | ✅ Yes |
| Source code | GitHub | ✅ Yes |

**Next notebooks:**
- `02_preprocessing.ipynb` — skull-stripping, resampling, normalisation
- `03_training.ipynb` — model training with checkpoint saving to Drive
- `04_evaluation.ipynb` — metrics computation and reporting

In [ ]:
print("✅ Notebook complete.")
print("")
print("Persistent storage contents:")
!ls -la {DRIVE_PROJECT_ROOT}